# PowerNet

> PowerNet is an model architecture for learning power allocation in wireless networks. It is designed to be simple and efficient, while still achieving state-of-the-art performance on the problem of power allocation in wireless networks.

In [1]:
#| default_exp models.power_net

In [2]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch
from torch import nn
from torch.nn import functional as F
from einops import rearrange
from torch.distributions import Bernoulli, Normal

from c3jepa_wm.utils import channel

## MLP

In [4]:
#| export
class PowerNetMLP(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, csi_dim, max_power):
        super().__init__()
        self.max_power = max_power
        self.embedding = nn.Embedding(num_embeddings, embedding_dim)
        
        # CSI is complex number → treat as 2 real values per agent
        self.net = nn.Sequential(
            nn.Linear(embedding_dim + csi_dim * 2, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
        )
        self.schedule_head = nn.Linear(64, 1)   # binary send/no send
        self.power_head = nn.Linear(64, 1)       # power level

    def forward(self, message_indices, csi):
        # message_idx: [B] integer indices
        # csi: [B, csi_dim] complex → pass as [B, csi_dim*2] real

        msg_embed = self.embedding(message_indices)      # [B, 49, embedding_dim]
        msg_flat = msg_embed.flatten(1)                  # [B, 49 * embedding_dim]
        csi_real = torch.view_as_real(csi).flatten(-2)   # [B, csi_dim*2]
        x = torch.cat([msg_flat, csi_real], dim=-1)
        features = self.net(x)
        
        schedule = torch.sigmoid(self.schedule_head(features))        # [B, 1] ∈ (0,1)
        power = torch.sigmoid(self.power_head(features)) * self.max_power  # [B, 1] ∈ (0, P_max)
        
        return schedule, power

## Transformer-based PowerNET

In [ ]:
#| export
class PowerNetMasked(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, csi_dim, max_power,
                 nhead=4, num_layers=2):
        super().__init__()
        self.max_power = max_power
        self.embedding_dim = embedding_dim

        # Message token embedding
        self.msg_embedding = nn.Embedding(num_embeddings, embedding_dim)

        # Project CSI (complex → 2 reals) to embedding_dim to use as query token
        self.csi_proj = nn.Linear(csi_dim * 2, embedding_dim)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=nhead,
            dim_feedforward=256,
            dropout=0.1,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Output heads — operate on the CSI query token output
        self.schedule_head = nn.Linear(embedding_dim, 1)
        self.power_head = nn.Linear(embedding_dim, 1)

    def forward(self, message_indices, csi):
        """
        message_indices: [B, 49] — same message for all neighbors
        csi: [B, num_neighbors, csi_dim] complex — different per neighbor
        returns:
            schedule: [B, num_neighbors, 1]
            power:    [B, num_neighbors, 1]
        """
        B, num_neighbors, _ = csi.shape

        # Embed message tokens: [B, 49, D]
        msg_tokens = self.msg_embedding(message_indices.view(B, -1))

        # Project CSI to embedding dim: [B, num_neighbors, D]
        csi_real = torch.view_as_real(csi).flatten(-2)      # [B, num_neighbors, csi_dim*2]
        csi_tokens = self.csi_proj(csi_real)                 # [B, num_neighbors, D]

        # Process each neighbor separately but efficiently using batch dim
        # Expand message tokens for each neighbor: [B*num_neighbors, 49, D]
        msg_expanded = msg_tokens.unsqueeze(1).expand(
            -1, num_neighbors, -1, -1
        ).reshape(B * num_neighbors, 49, self.embedding_dim)

        # CSI token as first token (acts as query over message)
        # [B*num_neighbors, 1, D]
        csi_expanded = csi_tokens.reshape(B * num_neighbors, 1, self.embedding_dim)

        # Concatenate: CSI token + 49 message tokens → sequence of 50
        sequence = torch.cat([csi_expanded, msg_expanded], dim=1)  # [B*N, 50, D]

        # Transformer: CSI token attends to message tokens
        out = self.transformer(sequence)                     # [B*N, 50, D]

        # Take output at CSI token position (index 0)
        csi_out = out[:, 0, :]                               # [B*N, D]

        # Output heads
        schedule = torch.sigmoid(
            self.schedule_head(csi_out)
        ).view(B, num_neighbors, 1)                          # [B, num_neighbors, 1]

        power = torch.sigmoid(
            self.power_head(csi_out)
        ).view(B, num_neighbors, 1) * self.max_power         # [B, num_neighbors, 1]

        return schedule, power
    

In [ ]:
#| hide
# B=8, 3 neighbors, 1 CSI value per neighbor
B = 8
neighbors= 3
powernet = PowerNetMasked(num_embeddings=256, embedding_dim=256, csi_dim=1, max_power=10.0)
obs = torch.randn(B, 3, 224, 224)  # [B, C, H, W]
message_indices = torch.randint(0, 256, (B, 7, 7))  # [B, 7, 7]
message_flat = message_indices.view(B, -1)                 # [B, 49]
csi = torch.randn(B, neighbors, 1, dtype=torch.complex64)          # [B, 3, 1]

schedule, power = powernet(message_flat, csi)
# schedule: [B, 3, 1] — per neighbor send/no-send
# power:    [B, 3, 1] — per neighbor power level

In [33]:
#| hide
power

tensor([[[3.6058],
         [3.7526],
         [3.4836]],

        [[4.5255],
         [3.5999],
         [3.5742]],

        [[5.5217],
         [3.5087],
         [3.6000]],

        [[4.6323],
         [3.3763],
         [4.6673]],

        [[5.2038],
         [3.3778],
         [3.3321]],

        [[4.3735],
         [3.2546],
         [4.6422]],

        [[4.6176],
         [5.6576],
         [4.2385]],

        [[4.8412],
         [3.2952],
         [3.5338]]], grad_fn=<MulBackward0>)

## Cross-Attention PowerNET

In [ ]:
#| export
import math
class PowerNet(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, csi_dim, max_power,
                 nhead=4, num_layers=2):
        super().__init__()
        self.max_power = max_power
        self.embedding_dim = embedding_dim

        self.msg_embedding = nn.Embedding(num_embeddings, embedding_dim)
        self.csi_proj = nn.Linear(csi_dim * 2, embedding_dim)
        self.log_power_std = nn.Parameter(
            torch.tensor(math.log(max_power / 4.0))
            )


        # Self-attention over message tokens first
        msg_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=nhead,
            dim_feedforward=256,
            dropout=0.1,
            batch_first=True
        )
        self.msg_transformer = nn.TransformerEncoder(msg_layer, num_layers=num_layers)

        # Cross-attention: CSI queries over message tokens
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=nhead,
            batch_first=True
        )

        self.schedule_head = nn.Linear(embedding_dim, 1)
        self.power_head = nn.Linear(2*embedding_dim, 1)

    @property
    def power_std(self):
        return self.log_power_std.exp().clamp(1e-4, self.max_power)

    def forward(self, message_indices, csi):
        """
        message_indices: [B, 49]
        csi: [B, num_neighbors, csi_dim] complex
        returns:
            schedule: [B, num_neighbors, 1]
            power:    [B, num_neighbors, 1]
        """
        B, num_neighbors, _ = csi.shape

        # 1. Embed and contextualize message tokens with self-attention
        msg_tokens = self.msg_embedding(message_indices)     # [B, 49, D]
        msg_ctx = self.msg_transformer(msg_tokens)           # [B, 49, D]

        # 2. Project CSI to query vectors
        csi_real = torch.view_as_real(csi).flatten(-2)       # [B, num_neighbors, csi_dim*2]
        csi_query = self.csi_proj(csi_real)                  # [B, num_neighbors, D]

        # 3. Cross-attention: each neighbor's CSI queries over message tokens
        # Expand msg_ctx for each neighbor
        msg_expanded = msg_ctx.unsqueeze(1).expand(
            -1, num_neighbors, -1, -1
        ).reshape(B * num_neighbors, 49, self.embedding_dim) # [B*N, 49, D]

        csi_expanded = csi_query.reshape(
            B * num_neighbors, 1, self.embedding_dim
        )                                                     # [B*N, 1, D]

        # CSI is query, message tokens are keys and values
        # "given my channel quality, what parts of this message matter?"
        attended, _ = self.cross_attn(
            query=csi_expanded,                              # [B*N, 1, D]
            key=msg_expanded,                                # [B*N, 49, D]
            value=msg_expanded                               # [B*N, 49, D]
        )                                                    # [B*N, 1, D]

        # out = attended.squeeze(1)                            # [B*N, D]

        attended_msg = attended.squeeze(1)           # [B*N, D] — message relevance per neighbor
        csi_expanded_flat = csi_expanded.squeeze(1)  # [B*N, D] — channel representation
        
        # 4. Per-neighbor decisions
        schedule = torch.sigmoid(
            self.schedule_head(attended_msg)         # what does the message contain?
        ).view(B, num_neighbors, 1)

        power_input = torch.cat([attended_msg, csi_expanded_flat], dim=-1)  # [B*N, 2D]
        power = torch.sigmoid(
            self.power_head(power_input)             # how hard to deliver this message?
        ).view(B, num_neighbors, 1) * self.max_power

        return schedule, power

    # def loss_fn(self, model, B, ctx_len, ctx_emb, ctx_act, msg_indices, tgt_emb, 
    #         schedule, power, csi_flat, pred_loss, lambda_value, lambda_pow, lambda_send, device):
    #     with torch.no_grad():
    #         # Baseline: prediction with no message
    #         received_msg = channel(
    #             schedule, power, msg_indices, csi_flat, device=device, no_comm=True
    #         )  # (B*T, 49)
    #         received_msg = rearrange(
    #             received_msg, "(b t) msg_dim -> b t msg_dim", b=B, t=ctx_len
    #         )  # (B, T, 49)
    #         ctx_msg = received_msg[:, :ctx_len]                # (B, T, 49) — raw indices, embedded inside predictor
            
    #         pred_emb_no_msg = model.predict(ctx_emb, ctx_act, ctx_msg)
    #         baseline_loss = (pred_emb_no_msg.detach() - tgt_emb).pow(2).mean()
    #         # How much did transmitting actually help?
    #         # Positive = transmission helped, Negative = transmission hurt
    #         comm_gain = baseline_loss - pred_loss  # scalar

    #         # logger.info(f"Computing perfect communication loss for quality metric.")
    #         msg_indices = rearrange(msg_indices, "(b t) msg_dim -> b t msg_dim", msg_dim=49, b=B, t=ctx_len)   # (B, T, 49)
    #         ctx_msg = msg_indices[:, :ctx_len] # (B, ctx_len, msg_dim)
    #         pred_emb_perfect_msg = self.model.predict(ctx_emb, ctx_act, ctx_msg) # pred with perfect comm.
    #         perfect_loss = (pred_emb_perfect_msg.detach() - tgt_emb).pow(2).mean()
    #         quality = perfect_loss - pred_loss

    #     s_mean = schedule.mean()   # (0,1) — how often we transmit
    #     p_mean = power.mean()      # (0, P_max) — average power used

    #     # 1. Reward scheduling when communication improves prediction
    #     reward_term = -lambda_value * comm_gain * s_mean

    #     # 2. Penalize power consumption (only when transmitting)
    #     power_term = lambda_pow * p_mean * s_mean

    #     # 3. Sparsity: penalize transmitting at all (encourages selective sending)
    #     sparsity_term = lambda_send * s_mean

    #     tx_loss = reward_term + power_term + sparsity_term
    #     return {
    #         "reward_term_loss": reward_term,
    #         "power_term_loss": power_term,
    #         "sparsity_term_loss": sparsity_term,
    #         "tx_loss": tx_loss,
    #         "base_line_loss": baseline_loss,
    #         "comm_gain": comm_gain,
    #         "schedule_mean": s_mean,
    #         "power_mean": p_mean,
    #         "quality": quality
    #     }
    
    def loss_fn(self, critic, reward, msg_indices, csi, schedule, power):
        """
        schedule: (B*T, N, 1) — sigmoid probabilities from PowerNet
        power:    (B*T, N, 1) — mean power from PowerNet
        """
        # ── Reward ────────────────────────────────────────────────────────
        # with torch.no_grad():
        #     pred_emb_no_msg = model.predict(ctx_emb, ctx_act, None)
        #     baseline_loss = (pred_emb_no_msg - tgt_emb).pow(2).mean()
        #     reward = baseline_loss - pred_loss                 # scalar

        # ── Critic ────────────────────────────────────────────────────────
        values = critic(msg_indices, csi)                      # (B*T, N, 1)
        reward_expanded = reward.expand_as(values)
        critic_loss = F.mse_loss(values, reward_expanded.detach())

        # Advantage per neighbor
        advantage = (reward_expanded - values).detach()        # (B*T, N, 1)

        # ── Schedule (discrete) — REINFORCE ───────────────────────────────
        schedule_dist = torch.distributions.Bernoulli(probs=schedule.clamp(1e-6, 1-1e-6))
        schedule_sample = (schedule > 0.5).float().detach()    # hard decision
        log_prob_schedule = schedule_dist.log_prob(schedule_sample)  # (B*T, N, 1)
        schedule_loss = -(advantage * log_prob_schedule).mean()

        # ── Power (continuous) — Gaussian policy ──────────────────────────
        power_dist = torch.distributions.Normal(
            loc=power,
            scale=self.power_std                               # learnable scalar std
        )
        power_sample = power_dist.rsample().clamp(0, self.max_power)
        log_prob_power = power_dist.log_prob(power_sample)     # (B*T, N, 1)

        # Power policy only matters when transmitting
        power_loss = -(advantage * log_prob_power * schedule_sample).mean()
        power_penalty = self.lambda_pow * (power * schedule_sample).mean()

        # ── Entropy bonuses ───────────────────────────────────────────────
        schedule_entropy = -self.lambda_entropy * schedule_dist.entropy().mean()
        power_entropy    = -self.lambda_entropy * power_dist.entropy().mean()

        # ── Total ─────────────────────────────────────────────────────────
        actor_loss = (schedule_loss + power_loss + 
                    power_penalty + schedule_entropy + power_entropy)
        tx_loss = actor_loss + self.lambda_critic * critic_loss

        return tx_loss, critic_loss, actor_loss


In [6]:
#| hide
# B=8, 3 neighbors, 1 CSI value per neighbor
B = 8
neighbors= 1
powernet = PowerNet(num_embeddings=256, embedding_dim=256, csi_dim=1, max_power=10.0)
obs = torch.randn(B, 3, 224, 224)  # [B, C, H, W]
message_indices = torch.randint(0, 256, (B, 7, 7))  # [B, 7, 7]
message_flat = message_indices.view(B, -1)                 # [B, 49]
csi = torch.randn(B, neighbors, 1, dtype=torch.complex64)          # [B, neighbors, 1]

schedule, power = powernet(message_flat, csi)
# schedule: [B, neighbors, 1] — per neighbor send/no-send
# power:    [B, neighbors, 1] — per neighbor power level

In [7]:
#| hide
powernet

PowerNet(
  (msg_embedding): Embedding(256, 256)
  (csi_proj): Linear(in_features=2, out_features=256, bias=True)
  (msg_transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (cross_attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
  )
  (schedule_head): Linear(in_features=2

In [ ]:
#| hide
power

tensor([[[5.4905]],

        [[5.5123]],

        [[5.5366]],

        [[5.5234]],

        [[5.5544]],

        [[5.5774]],

        [[5.5064]],

        [[5.5383]]], grad_fn=<MulBackward0>)

In [ ]:
#| export
class PowerCritic(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, csi_dim, nhead=4, num_layers=2):
        super().__init__()
        self.embedding_dim = embedding_dim

        self.msg_embedding = nn.Embedding(num_embeddings, embedding_dim)
        self.csi_proj = nn.Linear(csi_dim * 2, embedding_dim)

        # Self-attention over message tokens (same structure as actor)
        msg_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=nhead,
            dim_feedforward=256,
            dropout=0.1,
            batch_first=True
        )
        self.msg_transformer = nn.TransformerEncoder(msg_layer, num_layers=num_layers)

        # Cross-attention: CSI queries over message tokens
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=nhead,
            batch_first=True
        )

        # Value Head: Estimates the expected reward per neighbor given (CSI + Message)
        self.value_head = nn.Linear(2 * embedding_dim, 1)

    def forward(self, message_indices, csi):
        """
        message_indices: [B, 49]
        csi: [B, num_neighbors, csi_dim] complex
        returns:
            values: [B, num_neighbors, 1] - Expected reward baseline per neighbor
        """
        B, num_neighbors, _ = csi.shape

        # 1. Message Context
        msg_tokens = self.msg_embedding(message_indices)     # [B, 49, D]
        msg_ctx = self.msg_transformer(msg_tokens)           # [B, 49, D]

        # 2. CSI Query
        csi_real = torch.view_as_real(csi).flatten(-2)       # [B, num_neighbors, csi_dim*2]
        csi_query = self.csi_proj(csi_real)                  # [B, num_neighbors, D]

        # 3. Cross-Attention
        msg_expanded = msg_ctx.unsqueeze(1).expand(
            -1, num_neighbors, -1, -1
        ).reshape(B * num_neighbors, 49, self.embedding_dim) # [B*N, 49, D]

        csi_expanded = csi_query.reshape(
            B * num_neighbors, 1, self.embedding_dim
        )                                                     # [B*N, 1, D]

        attended, _ = self.cross_attn(
            query=csi_expanded,
            key=msg_expanded,
            value=msg_expanded
        )                                                     # [B*N, 1, D]

        attended_msg = attended.squeeze(1)                    # [B*N, D]
        csi_expanded_flat = csi_expanded.squeeze(1)            # [B*N, D]

        # 4. Value Estimation
        value_input = torch.cat([attended_msg, csi_expanded_flat], dim=-1)  # [B*N, 2D]
        
        # Linear layer outputs unconstrained value estimate per neighbor
        values = self.value_head(value_input).view(B, num_neighbors, 1)     # [B, num_neighbors, 1]

        return values

In [ ]:
#| hide
# B=8, 3 neighbors, 1 CSI value per neighbor
B = 8
neighbors= 1
powernet = PowerCritic(num_embeddings=256, embedding_dim=256, csi_dim=1)
obs = torch.randn(B, 3, 224, 224)  # [B, C, H, W]
message_indices = torch.randint(0, 256, (B, 7, 7))  # [B, 7, 7]
message_flat = message_indices.view(B, -1)                 # [B, 49]
csi = torch.randn(B, neighbors, 1, dtype=torch.complex64)          # [B, neighbors, 1]

out = powernet(message_flat, csi)
# schedule: [B, neighbors, 1] — per neighbor send/no-send
# power:    [B, neighbors, 1] — per neighbor power level

In [ ]:
# #| export
# def loss_fn(self, model, critic, ctx_len, ctx_emb, ctx_act,
#             msg_indices, csi, tgt_emb, schedule_logits, power, pred_loss):
    
#     with torch.no_grad():
#         # Actual reward: improvement over no-communication baseline
#         pred_emb_no_msg = model.predict(ctx_emb, ctx_act, None)
#         baseline_loss = (pred_emb_no_msg - tgt_emb).pow(2).mean()
#         reward = baseline_loss - pred_loss                     # scalar

#     # ── Critic ────────────────────────────────────────────────────────
#     # Critic estimates expected reward given state (msg + csi)
#     value = critic(msg_indices, csi)                           # (B*T, 1)
#     value_mean = value.mean()

#     # Critic loss: MSE between predicted value and actual reward
#     critic_loss = F.mse_loss(value_mean, reward.detach())

#     # ── Actor ─────────────────────────────────────────────────────────
#     # Advantage: did we do better than the critic expected?
#     with torch.no_grad():
#         advantage = reward - value_mean                        # scalar

#     # REINFORCE with advantage as baseline (reduces variance)
#     schedule_dist = torch.distributions.Bernoulli(logits=schedule_logits)
#     schedule_sample = (schedule_logits > 0).float()            # hard decision
#     log_prob = schedule_dist.log_prob(schedule_sample)         # (B*T*n, 1)

#     # Policy gradient loss
#     actor_loss = -(advantage * log_prob).mean()

#     # Power penalty: only when transmitting
#     power_loss = self.lambda_pow * (power * schedule_sample).mean()

#     # Entropy bonus: prevent collapse
#     entropy_bonus = -self.lambda_entropy * schedule_dist.entropy().mean()

#     # ── Total ─────────────────────────────────────────────────────────
#     tx_loss = actor_loss + power_loss + entropy_bonus + self.lambda_critic * critic_loss

#     return tx_loss, critic_loss, actor_loss

In [ ]:
# #| export

# def compute_powernet_ac_loss(actor, critic, predictor_net, ctx_len, ctx_emb, ctx_act, tgt_emb, pred_loss,
#                              message_indices, csi, input_tokens, power_penalty_coeff=0.01, exploration_std=1.0):
#     """
#     actor:             PowerNet instance
#     critic:            PowerCritic instance
#     predictor_net:     Frozen external metric network
#     message_indices:   [B, 49]
#     csi:               [B, num_neighbors, csi_dim]
#     input_tokens:      Raw message text/tokens required by your predictor_net
#     """
#     B, num_neighbors, _ = csi.shape
#     device = csi.device

#     csi_flat = rearrange(csi, "b n csi_dim -> (b n) csi_dim")  # [B*num_neighbors, csi_dim]

#     with torch.no_grad():
#             # Baseline: prediction with no message
#             received_msg = channel(
#                 schedule, power, message_indices, csi_flat, device=device, no_comm=True
#             )  # (B*T, 49)
#             received_msg = rearrange(
#                 received_msg, "(b t) msg_dim -> b t msg_dim", b=B, t=ctx_len
#             )  # (B, T, 49)
#             ctx_msg = received_msg[:, :ctx_len]                # (B, T, 49) — raw indices, embedded inside predictor
            
#             pred_emb_no_msg = predictor_net.predict(ctx_emb, ctx_act, ctx_msg)
#             baseline_loss = (pred_emb_no_msg.detach() - tgt_emb).pow(2).mean()
#             # How much did transmitting actually help?
#             # Positive = transmission helped, Negative = transmission hurt
#             # Predictor returns decoding success probability or rate performance per neighbor link
#             raw_rewards = baseline_loss - pred_loss  # scalar
#             # Apply competitive penalty to prevent the actor from over-allocating power
#             # power_penalty = power_penalty_coeff * actual_power_used
#             # # Target Return (G) = Objective Success Metric minus Resource Expense
#             # target_rewards = raw_rewards - power_penalty # [B, num_neighbors, 1]


#             # # logger.info(f"Computing perfect communication loss for quality metric.")
#             # msg_indices = rearrange(msg_indices, "(b t) msg_dim -> b t msg_dim", msg_dim=49, b=B, t=ctx_len)   # (B, T, 49)
#             # ctx_msg = msg_indices[:, :ctx_len] # (B, ctx_len, msg_dim)
#             # pred_emb_perfect_msg = predictor_net.predict(ctx_emb, ctx_act, ctx_msg) # pred with perfect comm.
#             # perfect_loss = (pred_emb_perfect_msg.detach() - tgt_emb).pow(2).mean()
#             # quality = perfect_loss - pred_loss

#     # =========================================================================
#     # 1. ACTOR FORWARD PASS & STOCHASTIC SAMPLING (Per Neighbor)
#     # =========================================================================
#     sched_probs, power_mu = actor(message_indices, csi) # Both shapes: [B, num_neighbors, 1]

#     # Create distributions for every separate neighbor link
#     sched_dist = Bernoulli(probs=sched_probs)
#     sched_action = sched_dist.sample() # [B, num_neighbors, 1] (0.0 or 1.0)
#     sched_log_prob = sched_dist.log_prob(sched_action)

#     # Power Exploration Distribution
#     # exploration_std controls how wildly the model guesses new power variations during training
#     power_dist = Normal(loc=power_mu, scale=exploration_std)
#     power_action = power_dist.sample()
#     # Clip sampled power to valid physically possible ranges [0, max_power]
#     power_action = torch.clamp(power_action, 0.0, actor.max_power)
#     power_log_prob = power_dist.log_prob(power_action)

#     # =========================================================================
#     # 2. CRITIC FORWARD PASS
#     # =========================================================================
#     # Critic estimates what reward it expects for EACH neighbor channel state
#     value_estimates = critic(message_indices, csi) # [B, num_neighbors, 1]

#     # =========================================================================
#     # 3. ENVIRONMENT EVALUATION (PREDICTOR INTERACTION - NO GRADIENTS)
#     # =========================================================================
#     with torch.no_grad():
#         # Mask power: If the model chose not to schedule (0.0), power used is 0.0
#         actual_power_used = power_action * sched_action # [B, num_neighbors, 1]

#         # --- COMPUTE PREDICTOR REWARD ---
#         # Reshape or iterate per neighbor depending on how your predictor accepts inputs.
#         # Assuming your predictor processes neighbor channels independently:
#         predictor_input = {
#             'tokens': input_tokens,                    # Original message attributes
#             'power': actual_power_used.squeeze(-1),    # [B, num_neighbors]
#             'csi': csi
#         }
        
#         # Predictor returns decoding success probability or rate performance per neighbor link
#         raw_rewards = predictor_net(predictor_input).detach() # Expected Shape: [B, num_neighbors, 1]

#         # Apply competitive penalty to prevent the actor from over-allocating power
#         power_penalty = power_penalty_coeff * actual_power_used
        
#         # Target Return (G) = Objective Success Metric minus Resource Expense
#         target_rewards = raw_rewards - power_penalty # [B, num_neighbors, 1]

#     # =========================================================================
#     # 4. ADVANTAGE & CRITIC LOSS
#     # =========================================================================
#     # Advantage = How much better/worse did this exploration step turn out?
#     # We detach the targets to prevent critic updates from rolling into the environment pipeline
#     advantages = target_rewards - value_estimates.detach() # [B, num_neighbors, 1]

#     # Critic learns via Mean Squared Error to mimic target rewards accurately
#     loss_critic = F.mse_loss(value_estimates, target_rewards)

#     # =========================================================================
#     # 5. ACTOR POLICY GRADIENT LOSS
#     # =========================================================================
#     # REINFORCE policy adjustments weighted by the Advantage
#     loss_sched_raw = - (sched_log_prob * advantages)
#     loss_sched = loss_sched_raw.mean()

#     # Power optimization logic:
#     # Only calculate gradients for power if the chosen action actually set schedule=True
#     loss_power_raw = - (power_log_prob * advantages)
#     mask = (sched_action == 1)
    
#     if mask.sum() > 0:
#         loss_power = loss_power_raw[mask].mean()
#     else:
#         loss_power = torch.tensor(0.0, device=device)

#     # =========================================================================
#     # 6. TOTAL MULTI-TASK COMBINED AC LOSS
#     # =========================================================================
#     # 0.5 is a standard hyperparameter scale coefficient for stabilizing Critic convergence
#     total_loss = loss_sched + loss_power + (0.5 * loss_critic)

#     return total_loss, loss_sched.item(), loss_power.item(), loss_critic.item(), raw_rewards.mean().item()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()